<a href="https://colab.research.google.com/github/jhdeov/audio-to-textgrid-batch-processor/blob/main/Long_Form_transcription_to_TextGrids.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Batch long-form transcription of speech to TextGrids with Whisper and Silero VAD

This notebook is designed to automate the transcription of large audio files, and convert the transcriptions to time-aligned SRTs TextGrids. It utilizes a Whisper model for speech recognition and the Silero Voice Activity Detector (VAD) for silence detection. This notebook is geared for linguists or language researchesr who want to transcribe audio files such as for an oral corpus.

The workflow of the script is as follows:
1) It takes a folder of audio files as input.
2) It detects the silence intervals in an audio file, using Silero Voice Activity Detector (VAD).
3) It breaks up the speech stream into separate non-silent chunks.
4) Each non-silent chunk passes through your transcription model to get transcribed
5) The individual chunk transcriptions are concatenated to create your final transcription
6) The output is saved as SRTs and TextGrids.

For step 4, you can plug in the repository name of a Whisper model from Hugging Face. Otherwise, if your model is not on Hugging Face or is not a Whisper model, you'll need to modify the code to set up the model in section 1.3


The only work you need to do is enter your folder path and file names in section 1

# 1. Preliminary steps that require your information

## 1.1. Mount Google Drive and access OS

Mount your Google Drive and be able to upload/download files.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from google.colab import files
import os

## 1.2 Set up audio files and directories

Provide the following variables:
* **audio_directory**: The full path to the Google Drive folder that contains your audio files. Make sure your path starts and ends with the '/' symbol. For example, `/content/drive/MyDrive/test/`
* **audio_names**: The list of filenames for your audio files. Write it as a Python list of strings. Make sure you include the extension for the files. For example `
[ "file1.wav" ,"file2.mp3" ]`

We then use these variables to create the output folder. The output folder contains SRTs and TextGrids. You likely only want to use the TextGrids.

Note: Two types of SRT files are created: original and cleaned. The cleaned SRTs files include silence intervals.

In [1]:
audio_directory = "/content/drive/MyDrive/Coding and data/General Armenian resources/armenianASR/lrec asr/evidentiality_recording/" # @param {"type":"string","placeholder":"Directory for audio files"}
assert audio_directory.startswith("/") and audio_directory.endswith("/")
audio_names = [  "diana.wav", "muraz.wav", "talya_erol.mp3", "Garen_demo.mp3", "nayra_ciftciyan.wav", "talya_ingiliz.mp3"                  ] # @param {"type":"raw","placeholder":"List of fIlenames for audio files"}


In [3]:
import os

root_directory_for_outputs = audio_directory +  "output/"

original_srt_directory = root_directory_for_outputs + "original_srts/"
os.makedirs(original_srt_directory, exist_ok=True)
cleaned_srt_directory = root_directory_for_outputs + "cleaned_srts/"
os.makedirs(cleaned_srt_directory, exist_ok=True)
textgrid_directory = root_directory_for_outputs + "textgrids/"
os.makedirs(textgrid_directory, exist_ok=True)

## 1.3 Set up Whisper as your transcription model

Set up the Whisper model that you will use to transcribe your audio files.
Provide the following variables

* **model_name**: Name of the model from Hugging Face.
* **language**: Language of your audio files.


In [ ]:
model_name = "Huseyin/whisper-large-v3-turkish-finetuned" # @param {"type":"string","placeholder":"Model name from Hugging Face"}
language = "turkish" # @param {"type":"string","placeholder":"Name of language"}


In [ ]:
import torch
from tqdm import tqdm
from transformers import WhisperForConditionalGeneration, WhisperProcessor,WhisperTokenizer,WhisperFeatureExtractor, pipeline
import os
import librosa

In [ ]:
device = 0 if torch.cuda.is_available() else "cpu"

# Initialize the ASR pipeline
pipe = pipeline(
    task="automatic-speech-recognition",
    model=model_name,
    chunk_length_s=30, # Recommended for longer audio files
    device=device,
)

# Load the WhisperProcessor to get decoder prompt IDs
processor = WhisperProcessor.from_pretrained(MODEL_NAME)

# Specify the desired language and task
lang = "turkish"
task = "transcribe"


# Set the forced decoder IDs in the model's configuration
pipe.model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(
    language=lang,
    task=task
)

Loading whisper model...


Loading weights:   0%|          | 0/1259 [00:01<?, ?it/s]

Passing `generation_config` together with generation-related arguments=({'return_timestamps'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


Whisper model complete.


# 2. Preliminary steps that do not require your information

## 2.1 Set up Silero

Set up the Silero package to find silence intervals in your audio files.

In [ ]:
!pip install -q torchaudio

SAMPLING_RATE = 16000

import torch
torch.set_num_threads(1)

from IPython.display import Audio



In [ ]:
USE_PIP = True # download model using pip package or torch.hub
USE_ONNX = False # change this to True if you want to test onnx model
if USE_ONNX:
    !pip install -q onnxruntime
if USE_PIP:
  !pip install -q silero-vad
  from silero_vad import (load_silero_vad,
                          read_audio,
                          get_speech_timestamps as get_speech_timestamps_silero,
                          save_audio,
                          VADIterator,
                          collect_chunks)
  model_silero = load_silero_vad(onnx=USE_ONNX)
else:
  model_silero, utils = torch.hub.load(repo_or_dir='snakers4/silero-vad',
                                model='silero_vad',
                                force_reload=True,
                                onnx=USE_ONNX)

  (get_speech_timestamps_silero,
  save_audio,
  read_audio,
  VADIterator,
  collect_chunks) = utils

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 36.1 MB/s eta 0:00:00


## 2.2 Import audio splicing packages

Import the package that will handle breaking down your individual audio files into multiple smaller audio segments.

In [ ]:
from pydub import AudioSegment

## 2.3 Set up workflow to create SRTs and TextGrids

We set up functions and packages to create SRT files, which we then convert to TextGrids.

In [ ]:
from datetime import timedelta

"""Function that converts the transcriptions of spliced audio segments into sections of an SRT"""
def convertChunkToSr(transcriptions):
  segmentId = 0
  srtFileText = ""
  for chunk in transcriptions:
        start,end = chunk['segment_start_ms']/1000,chunk['segment_end_ms']/1000 # transcriptions have times in ms and we want it in seconds
        text = chunk['text']
        # print(start,end)
        startTime = str(timedelta(seconds=start)).replace(".",",")
        endTime = str(timedelta(seconds=end)).replace(".",",")
        # print(startTime,startTime[0],startTime[1],endTime)
        if "," not in startTime: startTime = startTime+",000"
        if "," not in endTime: endTime = endTime+",000"
        if startTime[1]== ":" : startTime = '0' + startTime
        else: print('wtf',startTime)
        if endTime[1]== ":" : endTime = '0' + endTime
        # print(startTime,startTime[0],startTime[1],endTime)
        segmentId = segmentId +1
        segment = f"{segmentId}\n{startTime} --> {endTime}\n{text}\n\n"
        srtFileText = srtFileText + segment

  return srtFileText


We use the SrtToTextgrid repo from Github to convert SRTs to TextGrids.

In [ ]:
import locale
locale.getpreferredencoding = lambda: "UTF-8"

In [ ]:
! git clone https://github.com/rctatman/SrtToTextgrid

Cloning into 'SrtToTextgrid'...
remote: Enumerating objects: 49, done.
remote: Counting objects: 100% (22/22), done.
remote: Compressing objects: 100% (17/17), done.
remote: Total 49 (delta 8), reused 17 (delta 5), pack-reused 27 (from 1)
Receiving objects: 100% (49/49), 4.75 MiB | 18.17 MiB/s, done.
Resolving deltas: 100% (15/15), done.


# 3. Batch Silero+Transcription

We run Silero and the transciber to process all your audio files in a sequence. You can check the progress for your current files. The output transcriptions are saved in the output folder.

In [ ]:
for audio_name in audio_names:
  # Get timestamps that have sound in them
  print("Working on file: " + audio_name)
  audio_filepath = audio_directory + audio_name
  audio_name_without_extension = audio_name.split(".")[0]
  audio_extension = audio_name.split(".")[-1]
  wav = read_audio(audio_filepath, sampling_rate=SAMPLING_RATE)
  timestamp_chunks = get_speech_timestamps_silero(wav, model_silero, sampling_rate=SAMPLING_RATE, return_seconds=True)

  transcriptions = []
  audio = AudioSegment.from_file(audio_filepath)

  num_segments = len(timestamp_chunks)
  print("Number of segments: ", num_segments)
  count = 0
  with tqdm(total=num_segments, desc="Transcribing Segments") as pbar:
    count = count + 1
    for i, times in enumerate(timestamp_chunks):
      start_ms = times['start']*1000 # Silero uses seconds, while pydup uses milliseconds so must convert manually
      end_ms = times['end']*1000
      print(start_ms,end_ms)
      segment = audio[start_ms:end_ms]
      temp_segment_path = f"temp_segment_{i}.{audio_extension}"
      segment.export(temp_segment_path, format=audio_extension)

      audio_array, sample_rate = librosa.load(temp_segment_path, sr=SAMPLING_RATE)
      result = pipe( audio_array,batch_size=8, )

      transcriptions.append({
                "segment_start_ms": start_ms,
                "segment_end_ms": end_ms,
                "text": normalize_transcription(result["text"])
            })
      # files.download(temp_segment_path)
      # print(transcriptions[-1])
      os.remove(temp_segment_path) # Clean up temporary file
      pbar.update(1)

  print("")
  print("Converting to SRT and TextGrid")
  original_srt = original_srt_directory + audio_name_without_extension + ".srt"
  f = open(original_srt, "w", encoding="utf-8")
  f.write(convertChunkToSr(transcriptions))
  # files.download(original_srt)
  f.close()

  cleaned_srt = cleaned_srt_directory + audio_name_without_extension + ".srt"
  textgrid_file = textgrid_directory + audio_name_without_extension + ".TextGrid"

  ! python3 SrtToTextgrid/SilentIntervalSRT.py "{original_srt}" "{cleaned_srt}"
  # files.download(cleaned_srt_path)
  ! python3 SrtToTextgrid/SrtToTextgrid.py "{cleaned_srt}" "{textgrid_file}"



Working on file: aslin
Number of segments:  191


Transcribing Segments:   0%|          | 0/191 [00:00<?, ?it/s]

600.0 2600.0


Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> will take precedence. Please check the docstring of <class 'tr

3600.0 4200.0


...

306700.0 309100.0


Transcribing Segments:  48%|████▊     | 101/210 [19:24<09:23,  5.17s/it]

309400.0 310000.0


Transcribing Segments:  49%|████▊     | 102/210 [19:26<07:33,  4.20s/it]

310200.0 311700.0


# end session

In [ ]:
from google.colab import runtime
runtime.unassign()